# Your first notebook on HPE Private Cloud AI

This notebook checks that your notebook server can reach a model served by **HPE MLIS** (the **Model Endpoints** in HPE AI Essentials 1.9.x), then runs a small **LangGraph** agent on it.

**You will need** (ask your trainer or administrator if you do not have them):

1. The **endpoint URL** of the LLM endpoint. The cell in Step 3 is pre-filled with the lab endpoint `llm-llama8b-1`, so you can press Enter to accept it or paste your own.
2. Optional: the endpoint URL of an **embedding** endpoint (pre-filled with `embedder-llama8b-1`).
3. Optional: a **token**, only if your endpoint requires one. The lab endpoints do not.

You do not need to type the **model name**. The notebook asks the endpoint which model it serves.

**How to use it:** run one cell at a time with `Shift + Enter`. Read the "Expected output" note under each step before you move on.

Nothing in this notebook stores a token. Do not paste one into a cell.

## Step 1. Check where you are running

This cell prints facts about the machine your kernel runs on. Your code runs inside the platform, not on your laptop.

**Expected output:** a Python version of 3.10 or later, a hostname that looks like a notebook pod name, and the path of your working folder.

In [ ]:
import os
import platform
import shutil
import sys

print("Python        :", sys.version.split()[0])
print("Platform      :", platform.platform())
print("Hostname      :", platform.node())
print("Working folder:", os.getcwd())
print("nvidia-smi    :", "found" if shutil.which("nvidia-smi") else "not found (normal: the model runs on MLIS, not in this notebook)")

## Step 2. Install the packages

`%pip install` installs into the notebook's base environment. Packages installed this way are **removed when the notebook server restarts**, so this cell must be re-run after a restart. `langchain-openai` is the LangChain client for OpenAI-compatible servers, which is what MLIS endpoints are. Installing needs access to a package index; it does not work in air-gapped environments.

**Expected output:** a short pause and no red error text. If a later cell says `ModuleNotFoundError`, restart the kernel (Kernel menu, Restart Kernel) and continue from Step 3.

In [ ]:
%pip install --quiet langchain-openai langgraph requests

## Step 3. Enter your endpoint details

The cell reads each setting from an environment variable if one exists (for the embedding endpoint, an empty value means "skip"), and otherwise asks you. **Press Enter to accept the value shown in square brackets.** Type `skip` at the embedding prompt to skip Step 6. The token prompt hides what you type; press Enter if your endpoint needs no token.

Where to find the values in HPE AI Essentials 1.9.x:

- **Endpoint URL:** Gen AI, **Model Endpoints**. Copy the value in the **Endpoint** column. The endpoint must show the status **Ready**; an endpoint that is still **Deploying** has no URL yet.
- **Model name:** not needed. Step 4 asks the endpoint for it. To force a particular model, set `MLIS_LLM_MODEL` (or `MLIS_EMB_MODEL`).

Paste the endpoint as shown. If it ends in `/chat/completions` or `/embeddings`, the cell trims that part. Step 4 works out whether `/v1` is needed.

**Expected output:** a summary of what was entered, with any token hidden.

In [ ]:
import getpass
import os

# Lab defaults: the two endpoints shown on the Model Endpoints screen.
# Change them here, at the prompt, or with the environment variables in the README.
DEFAULT_LLM_URL = "https://llm-llama8b-1.project-user-kiran-kumar-m.serving.labpcaidev.pcaicoe.com"
DEFAULT_EMB_URL = "https://embedder-llama8b-1.project-user-kiran-kumar-m.serving.labpcaidev.pcaicoe.com"


def ask(name, prompt, default="", secret=False):
    """Read a setting from an environment variable, or ask for it (Enter keeps the default)."""
    if name in os.environ:
        return os.environ[name].strip()
    shown = f"{prompt} [{default}]: " if default else f"{prompt}: "
    value = (getpass.getpass(shown) if secret else input(shown)).strip()
    return value or default


def clean_base_url(url):
    """Remove a trailing slash and any /chat/completions or /embeddings suffix."""
    url = url.strip().rstrip("/")
    for suffix in ("/chat/completions", "/embeddings"):
        if url.endswith(suffix):
            url = url[: -len(suffix)]
    return url


LLM_BASE_URL = clean_base_url(ask("MLIS_LLM_BASE_URL", "LLM endpoint URL", DEFAULT_LLM_URL))
if not LLM_BASE_URL:
    raise ValueError("MLIS_LLM_BASE_URL is required")
LLM_MODEL = os.environ.get("MLIS_LLM_MODEL", "").strip()
TOKEN = ask("MLIS_DEPLOY_TOKEN", "Token (hidden, press Enter if the endpoint needs none)", secret=True)

emb_url = ask("MLIS_EMB_BASE_URL", "Embedding endpoint URL (type skip to skip)", DEFAULT_EMB_URL)
EMB_BASE_URL = "" if emb_url.lower() == "skip" else clean_base_url(emb_url)
EMB_MODEL = os.environ.get("MLIS_EMB_MODEL", "").strip() if EMB_BASE_URL else ""
EMB_TOKEN = (os.environ["MLIS_EMB_TOKEN"].strip() if "MLIS_EMB_TOKEN" in os.environ else TOKEN) if EMB_BASE_URL else ""

print("LLM endpoint      :", LLM_BASE_URL)
print("LLM model         :", LLM_MODEL or "(will be detected in Step 4)")
print("Token entered     :", f"yes ({len(TOKEN)} characters)" if TOKEN else "no (endpoint called without a token)")
print("Embedding endpoint:", EMB_BASE_URL or "skipped")

## Step 4. Find the model and make one raw call

This uses plain `requests`, so you can see exactly what goes over the network.

1. `GET <endpoint>/models` asks the endpoint which model it serves and confirms the URL is right. MLIS endpoints follow the OpenAI-compatible layout, which puts the API under `/v1`, so the helper tries `<endpoint>/v1` first and then `<endpoint>`.
2. `POST <api root>/chat/completions` sends one question. If you entered a token, it goes in an `Authorization: Bearer` header; otherwise no header is sent.

Some models (for example Qwen3) write a reasoning block inside `<think>` tags before the answer. The helper `strip_reasoning` removes it so that later steps see only the answer.

**Expected output:** the API root, the model name, the word `ready` and a token-usage dictionary. If you get an error, read the **Hint** line in the message first.

In [ ]:
import re
import requests

HINTS = {
    401: "The endpoint wants a token, or the token is wrong. Enter a valid token in Step 3.",
    403: "The token is not allowed for this endpoint, or it has expired.",
    404: "Wrong URL path or model name. Copy the Endpoint value from Gen AI, Model Endpoints.",
    429: "Too many requests. Other participants share this endpoint. Wait a few seconds and retry.",
    502: "Gateway error. The endpoint may still be starting.",
    503: "The endpoint is not ready (starting, scaled to zero, or overloaded). Check its status in Model Endpoints.",
}


def strip_reasoning(text):
    """Remove a leading <think>...</think> block that some models emit."""
    return re.sub(r"<think>.*?(?:</think>|$)", "", text or "", flags=re.DOTALL).strip()


def request_json(method, url, token, payload=None, timeout=90):
    headers = {"Content-Type": "application/json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    resp = requests.request(method, url, headers=headers, json=payload, timeout=timeout)
    if not resp.ok:
        hint = HINTS.get(resp.status_code, "See the response body below.")
        raise RuntimeError(f"HTTP {resp.status_code} from {url}\nHint: {hint}\nBody: {resp.text[:500]}")
    return resp.json()


def post_json(url, token, payload, timeout=90):
    return request_json("POST", url, token, payload, timeout)


def discover(base_url, token, label):
    """Find the API root (the URL that answers GET /models) and the model names served there."""
    root = clean_base_url(base_url)
    candidates = [root] if root.endswith("/v1") else [f"{root}/v1", root]
    problems = []
    for api_root in candidates:
        try:
            data = request_json("GET", f"{api_root}/models", token, timeout=30)
            return api_root, [m["id"] for m in data.get("data", []) if "id" in m]
        except (RuntimeError, requests.RequestException, ValueError) as err:
            problems.append(" | ".join(str(err).splitlines()[:2])[:300])
    raise RuntimeError(f"Could not reach the {label} endpoint {root}\n" + "\n".join(problems))


def pick_model(requested, served, label, env_name):
    """Use the requested model if given, otherwise the first model the endpoint serves."""
    if requested:
        if served and requested not in served:
            print(f"Warning: {requested!r} is not in the list served by the {label} endpoint: {served}")
        return requested
    if not served:
        raise ValueError(f"The {label} endpoint listed no models. Set {env_name} to the model name.")
    if len(served) > 1:
        print(f"The {label} endpoint serves {len(served)} models; using the first. Set {env_name} to choose another.")
    return served[0]


LLM_API, llm_served = discover(LLM_BASE_URL, TOKEN, "LLM")
LLM_MODEL = pick_model(LLM_MODEL, llm_served, "LLM", "MLIS_LLM_MODEL")
print("LLM API root  :", LLM_API)
print("LLM model     :", LLM_MODEL)

reply = post_json(
    f"{LLM_API}/chat/completions",
    TOKEN,
    {
        "model": LLM_MODEL,
        "messages": [{"role": "user", "content": "Reply with the single word: ready"}],
        "temperature": 0,
        "max_tokens": 256,
    },
)
print("Answer:", strip_reasoning(reply["choices"][0]["message"]["content"]))
print("Usage :", reply.get("usage"))

## Step 5. Make the same call through LangChain

Agent frameworks talk to the same endpoint. LangChain has no separate class for MLIS, because MLIS endpoints speak the OpenAI-compatible protocol. `ChatOpenAI` is simply the LangChain client for that protocol: `base_url` points it at your MLIS endpoint, so **the request goes only to MLIS and nothing is sent to OpenAI**. The `langchain-openai` package name is a naming detail, not a dependency on the OpenAI service.

**Expected output:** a one-sentence answer and a usage summary.

**If this step fails but Step 4 worked:** the most common cause is a certificate that the Python HTTP client does not trust (private clouds often use an internal certificate authority). Ask your administrator for the CA file, then run `import os; os.environ['SSL_CERT_FILE'] = os.environ['REQUESTS_CA_BUNDLE'] = '/path/to/ca.pem'` in a cell before Step 4 and re-run from there.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    base_url=LLM_API,
    # The client insists on a non-empty key. With a token it is sent as
    # "Authorization: Bearer <token>"; without one, the placeholder is ignored by open endpoints.
    api_key=TOKEN or "not-needed",
    model=LLM_MODEL,
    temperature=0,
    max_tokens=512,
    timeout=90,
    max_retries=1,
)

answer = llm.invoke("In one sentence, what is an AI agent?")
print(strip_reasoning(answer.content))
print("Usage:", answer.usage_metadata)

## Step 6. Call the embedding model (optional)

Retrieval features such as long-term memory turn text into vectors. This step finds the embedding model the same way as Step 4 and calls `<api root>/embeddings`. NVIDIA retrieval embedding models such as `nv-embedqa-e5-v5` usually expect an `input_type` field (`query` or `passage`); the helper retries without it if the server rejects it.

**Expected output:** the API root, the model name, the vector length (for example 1024) and the first five values. If you skipped the embedding endpoint in Step 3, the cell says so and you can move on.

In [ ]:
def embed(texts, input_type="query"):
    payload = {"model": EMB_MODEL, "input": texts, "encoding_format": "float"}
    url = f"{EMB_API}/embeddings"
    try:
        return post_json(url, EMB_TOKEN, {**payload, "input_type": input_type})
    except RuntimeError as err:
        if "HTTP 400" in str(err) or "HTTP 422" in str(err):
            return post_json(url, EMB_TOKEN, payload)
        raise


if EMB_BASE_URL:
    EMB_API, emb_served = discover(EMB_BASE_URL, EMB_TOKEN, "embedding")
    EMB_MODEL = pick_model(EMB_MODEL, emb_served, "embedding", "MLIS_EMB_MODEL")
    print("Embedding API root:", EMB_API)
    print("Embedding model   :", EMB_MODEL)
    out = embed(["reset a user password"])
    vector = out["data"][0]["embedding"]
    print("Vector length:", len(vector))
    print("First 5 values:", [round(v, 4) for v in vector[:5]])
else:
    print("Skipped: no embedding endpoint was configured in Step 3.")

## Step 7. Run a small LangGraph agent

This is a two-path graph for the IT incident use case:

```
START -> classify -> (high severity)  -> escalate -> END
                  -> (other severity) -> resolve  -> END
```

- `classify` asks the model for a severity: low, medium or high.
- `route` decides the next node from the state. Routing on a plain value in the state keeps the decision deterministic.
- `escalate` stops and asks for a human. No system change is made.
- `resolve` asks the model for a first-line fix.

**Expected output:** a Mermaid diagram of the graph as text, then one result per incident. A small model can misjudge severity, so treat the classification as a starting point for discussion.

In [ ]:
import re
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph


class IncidentState(TypedDict, total=False):
    incident: str
    severity: str
    action: str


def classify(state: IncidentState) -> dict:
    prompt = (
        "Classify the severity of this IT incident as exactly one word: low, medium or high.\n"
        f"Incident: {state['incident']}"
    )
    text = strip_reasoning(llm.invoke(prompt).content).lower()
    match = re.search(r"\b(high|medium|low)\b", text)
    return {"severity": match.group(1) if match else "medium"}


def route(state: IncidentState) -> Literal["escalate", "resolve"]:
    return "escalate" if state["severity"] == "high" else "resolve"


def resolve(state: IncidentState) -> dict:
    prompt = f"Give a two-sentence first-line fix for this IT incident: {state['incident']}"
    return {"action": strip_reasoning(llm.invoke(prompt).content)}


def escalate(state: IncidentState) -> dict:
    return {"action": "ESCALATED: waiting for human approval before any system change."}


graph = StateGraph(IncidentState)
graph.add_node("classify", classify)
graph.add_node("resolve", resolve)
graph.add_node("escalate", escalate)
graph.add_edge(START, "classify")
graph.add_conditional_edges("classify", route, {"escalate": "escalate", "resolve": "resolve"})
graph.add_edge("resolve", END)
graph.add_edge("escalate", END)
app = graph.compile()

print(app.get_graph().draw_mermaid())

In [ ]:
incidents = [
    "The payments API is down for all customers and the error rate is 100 percent.",
    "A user in the finance office cannot see the new shared printer.",
]

results = []
for text in incidents:
    final = app.invoke({"incident": text})
    results.append(final)
    print("Incident:", final["incident"])
    print("Severity:", final["severity"])
    print("Action  :", final["action"])
    print("-" * 60)

## Step 8. Save the results

Files on your private notebook volume are still there after the server is stopped and started again. Ask your trainer which folder that is; if your working folder is not on it, save the file there instead. The token is never written.

**Expected output:** the full path of the saved JSON file. You can see it in the JupyterLab file browser on the left.

In [ ]:
import datetime
import json
import os

path = f"pcai_first_run_{datetime.datetime.now():%Y%m%d_%H%M%S}.json"
with open(path, "w") as handle:
    json.dump({"model": LLM_MODEL, "results": results}, handle, indent=2)

print("Saved:", os.path.abspath(path))

## Step 9. Finish

1. Save the notebook (File menu, Save Notebook).
2. If your trainer asks for it, use Kernel, Restart Kernel and Run All Cells to prove it runs from a clean start.
3. When you are done for the day, return to the **Notebook Servers** screen in HPE AI Essentials and **stop** your server to release CPU and memory. Your files stay in your volume.

### Quick troubleshooting

| What you see | Likely cause | What to do |
|---|---|---|
| `HTTP 401` or `403` | The endpoint requires a token, or the token is wrong or expired | Enter a valid token in Step 3 |
| `HTTP 404`, or "Could not reach the endpoint" | Wrong URL or model name | Copy the **Endpoint** value again from Gen AI, Model Endpoints; the notebook tries `/v1` for you |
| `HTTP 503`, timeout | Endpoint is **Deploying**, scaled to zero, or busy | Check the status in Model Endpoints (it must be **Ready**); wait and retry |
| SSL or certificate error | Internal certificate authority not trusted | Ask your administrator for the CA file and set `SSL_CERT_FILE` and `REQUESTS_CA_BUNDLE` (see Step 5) |
| `ModuleNotFoundError` | Kernel restarted, so `%pip` installs were removed | Re-run Step 2, restart the kernel, continue |
| "listed no models" | The endpoint does not implement `GET /models` | Set `MLIS_LLM_MODEL` (or `MLIS_EMB_MODEL`) to the model name and re-run Step 3 |
| Answer contains `<think>` text | Reasoning model | Already handled by `strip_reasoning`; keep using it |